## Import Data

In [1]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots
import streamlit as st

## Clean & Set Up Data

### Generate DF

In [2]:
# import .tsv as .csv
df = pd.read_csv('2025_specimen_time_series_events_no_phi.tsv', sep='\t')

In [3]:
df.head()

,accession_id,pat_enc_csn_id,pat_mrn_id,barcode,tube_id,specimen_id,test_code,test_performing_dept,test_performing_location,event_street,event_source,event_type,event_dt
0,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,TSH,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
1,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,BASIC,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
2,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,FT4,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
3,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,TSH,UCHM,ULAB,Hospital,order,test_collected_dt,2025-01-07T16:21:00Z
4,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,BASIC,UCHM,ULAB,Hospital,order,test_collected_dt,2025-01-07T16:21:00Z


In [4]:
df["event_source"].value_counts()

event_source
order           1647494
tube_tracker     527634
Name: count, dtype: int64

Reshape to long format (one row per specimen-event) - this way, we can easily compute durations to each event type & compare across specimens

In [5]:
# make sure event_dt is datetime
df['event_dt'] = pd.to_datetime(df['event_dt'])

In [6]:
# pivot to get one column per event type
# if there are multiple events of the same type for a specimen, we take 
# the earliest one (e.g. earliest resulted_dt)
df['test_id'] = df['accession_id'].astype(str) + "_" + df['test_code']

timeline = df.pivot_table(
    index='test_id',
    columns='event_type',
    values='event_dt',
    aggfunc='min'  # in case of duplicates, take earliest
).reset_index()

In [7]:
timeline

event_type,test_id,AdultER,BCSC,BCSC -POCT,BCSC-Heme,Blood Bank,C1,CANHC,CENTRIFUGE,CWS,...,YMBD Remote,cancellation_dt,p512 INFTY,test_collected_dt,test_max_resulted_dt,test_max_verified_dt,test_min_resulted_dt,test_min_verified_dt,test_ordered_dt,test_receipt_dt
0,00007d9883_25HD,NaT,NaT,2025-02-18 12:06:00+00:00,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-02-18 11:59:00+00:00,2025-02-18 16:04:00+00:00,2025-02-18 16:08:00+00:00,2025-02-18 16:04:00+00:00,2025-02-18 16:08:00+00:00,2025-02-18 11:58:00+00:00,2025-02-18 12:06:00+00:00
1,00007d9883_ALSUR,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,2025-02-18 12:02:00+00:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
2,0000b7bd82_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-04-02 11:39:00+00:00,2025-04-02 18:01:00+00:00,2025-04-02 18:15:00+00:00,2025-04-02 18:01:00+00:00,2025-04-02 18:15:00+00:00,2025-04-02 11:38:00+00:00,2025-04-02 17:38:00+00:00
3,0000ec2d4a_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-09-19 14:53:00+00:00,2025-09-20 01:58:00+00:00,2025-09-20 02:04:00+00:00,2025-09-20 01:58:00+00:00,2025-09-20 02:04:00+00:00,2025-09-19 14:52:00+00:00,2025-09-20 01:41:00+00:00
4,00014dfb63_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-06-16 15:20:00+00:00,2025-06-16 16:32:00+00:00,2025-06-16 16:31:00+00:00,2025-06-16 16:32:00+00:00,2025-06-16 16:31:00+00:00,2025-06-16 15:15:00+00:00,2025-06-16 15:50:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249461,fffc9c609a_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-02-05 15:25:00+00:00,2025-02-05 19:56:00+00:00,2025-02-05 20:00:00+00:00,2025-02-05 19:56:00+00:00,2025-02-05 20:00:00+00:00,2025-02-05 15:20:00+00:00,2025-02-05 15:40:00+00:00
249462,fffd4c2fe4_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-12-11 13:34:00+00:00,2025-12-11 16:24:00+00:00,2025-12-11 16:44:00+00:00,2025-12-11 16:24:00+00:00,2025-12-11 16:44:00+00:00,2025-12-11 13:30:00+00:00,2025-12-11 16:09:00+00:00
249463,fffe35a26c_GLUCM,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-09-24 16:03:00+00:00,2025-09-24 16:04:00+00:00,2025-09-24 16:04:00+00:00,2025-09-24 16:03:00+00:00,2025-09-24 16:03:00+00:00,2025-09-24 16:04:00+00:00,2025-09-24 16:03:00+00:00
249464,fffe675962_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-02-06 16:10:00+00:00,2025-02-06 22:36:00+00:00,2025-02-06 22:43:00+00:00,2025-02-06 22:36:00+00:00,2025-02-06 22:43:00+00:00,2025-02-06 16:07:00+00:00,2025-02-06 22:18:00+00:00


### Explore timeline DF

In [8]:
timeline.shape

(249466, 57)

In [9]:
# creat test_code for timeline by splitting test_id
timeline['test_code'] = timeline['test_id'].apply(lambda x: x.split('_')[1])

In [10]:
# how many unique test codes in the DF?
unique_test_codes = timeline['test_code'].nunique()
print(f"Number of unique test codes: {unique_test_codes}")

Number of unique test codes: 1022


In [11]:
# how many unique test ids in the DF?
unique_test_ids = timeline['test_id'].nunique()
print(f"Number of unique test ids: {unique_test_ids}")

Number of unique test ids: 249466


In [12]:
# timeline.columns

In [13]:
# Ensure datetime dtype
timeline['test_ordered_dt'] = pd.to_datetime(timeline['test_ordered_dt'], errors='coerce')

# Weekday = 0–4, Weekend = 5–6
timeline['day_of_week'] = timeline['test_ordered_dt'].dt.weekday  # Monday=0
timeline['day_type'] = timeline['day_of_week'].apply(lambda x: 'weekday' if x < 5 else 'weekend')

In [14]:
# what is the relative time of each event compared to the first event (test_ordered_dt) for each test_id?
relative_cols = [
    'test_ordered_dt', 'test_collected_dt', 'test_receipt_dt',
    'test_min_resulted_dt', 'test_max_resulted_dt',
    'test_min_verified_dt', 'test_max_verified_dt'
]

# restore test_ordered_dt from timeline in case it was overwritten in a prior run
timeline['test_ordered_dt'] = timeline['test_id'].map(
    timeline.set_index('test_id')['test_ordered_dt']
)

# ensure datetime dtype before subtraction
for col in relative_cols:
    timeline[col] = pd.to_datetime(timeline[col], utc=True, errors='coerce')

# create new relative columns (do not overwrite original datetime columns)
for col in relative_cols:
    timeline[col + '_relative'] = (
        timeline[col] - timeline['test_ordered_dt']
    ).dt.total_seconds() / 3600  # hours since ordered

    # round relative times to 2 decimal places
    timeline[col + '_relative'] = timeline[col + '_relative'].round(2)

# timeline.head()


In [15]:
timeline.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249466 entries, 0 to 249465
Data columns (total 67 columns):
 #   Column                         Non-Null Count   Dtype              
---  ------                         --------------   -----              
 0   test_id                        249466 non-null  object             
 1   AdultER                        997 non-null     datetime64[ns, UTC]
 2   BCSC                           28 non-null      datetime64[ns, UTC]
 3   BCSC -POCT                     3851 non-null    datetime64[ns, UTC]
 4   BCSC-Heme                      38 non-null      datetime64[ns, UTC]
 5   Blood Bank                     1301 non-null    datetime64[ns, UTC]
 6   C1                             55 non-null      datetime64[ns, UTC]
 7   CANHC                          3 non-null       datetime64[ns, UTC]
 8   CENTRIFUGE                     1124 non-null    datetime64[ns, UTC]
 9   CWS                            17848 non-null   datetime64[ns, UTC]
 10  Cancer C

### Create "non-canceled" orders DF

- Only use "complete" tests (no cancellation data)
- Make a copy of the DF, using cols that were not cancelled at any point (so cancellation_dt = null)

In [16]:
# make a df that excludes cancelled orders - we want to look at specimen
# timelines that are "complete" (no cancellations)

# if an order was cancelled, it would have a cancellation_dt, but no resulted_dt or verified_dt
# so only keep the ones where cancellation_dt is null
no_cancel_df = timeline[timeline['cancellation_dt'].isna()].copy()

# no_cancel_df

In [17]:
# timeline - no_cancel --> how many cancelled orders did we remove?
num_rows_removed = timeline.shape[0] - no_cancel_df.shape[0]
print(f"Number of orders removed due to cancellation: {num_rows_removed}")

Number of orders removed due to cancellation: 29063


In [18]:
no_cancel_df.columns
# no_cancel_df.describe()
# no_cancel_df.head(10)

Index(['test_id', 'AdultER', 'BCSC', 'BCSC -POCT', 'BCSC-Heme', 'Blood Bank',
       'C1', 'CANHC', 'CENTRIFUGE', 'CWS', 'Cancer Cntr', 'CardioV Ctr',
       'Cent-Reflab', 'Chemistry', 'DFLA Remote', 'EAAHC', 'Flow',
       'Hem/Coa/Urn', 'INFTY Rcv', 'IOM_C', 'IOM_H', 'Livonia', 'NCRC HLA',
       'NCRC Immuno', 'NCRC Micro', 'NCRC SpChem', 'NCRC SpQFTB', 'NLNC SP',
       'NOT SENDOUT', 'NoRcv Setup', 'Northv Rmt', 'Northv-poc', 'Northville',
       'P512', 'P612', 'PedsER', 'RIM', 'SENDOUTS', 'SP CoreAuto',
       'Spec. Proc.', 'Taubman1', 'Taubman3', 'UCW', 'UHS', 'West POC',
       'West Recv', 'West Rmote', 'YMBD Remote', 'cancellation_dt',
       'p512 INFTY', 'test_collected_dt', 'test_max_resulted_dt',
       'test_max_verified_dt', 'test_min_resulted_dt', 'test_min_verified_dt',
       'test_ordered_dt', 'test_receipt_dt', 'test_code', 'day_of_week',
       'day_type', 'test_ordered_dt_relative', 'test_collected_dt_relative',
       'test_receipt_dt_relative', 'test_min_res

In [19]:
# get the test code from the test_id column
no_cancel_df['test_code'] = no_cancel_df['test_id'].apply(lambda x: x.split('_')[1])

no_cancel_df.head()

event_type,test_id,AdultER,BCSC,BCSC -POCT,BCSC-Heme,Blood Bank,C1,CANHC,CENTRIFUGE,CWS,...,test_code,day_of_week,day_type,test_ordered_dt_relative,test_collected_dt_relative,test_receipt_dt_relative,test_min_resulted_dt_relative,test_max_resulted_dt_relative,test_min_verified_dt_relative,test_max_verified_dt_relative
0,00007d9883_25HD,NaT,NaT,2025-02-18 12:06:00+00:00,NaT,NaT,NaT,NaT,NaT,NaT,...,25HD,1.0,weekday,0.0,0.02,0.13,4.10,4.10,4.17,4.17
2,0000b7bd82_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,25HD,2.0,weekday,0.0,0.02,6.00,6.38,6.38,6.62,6.62
3,0000ec2d4a_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,25HD,4.0,weekday,0.0,0.02,10.82,11.10,11.10,11.20,11.20
4,00014dfb63_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,25HD,0.0,weekday,0.0,0.08,0.58,1.28,1.28,1.27,1.27
5,000168b72f_ACTMN,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,ACTMN,1.0,weekday,0.0,1.02,1.35,1.80,1.80,1.93,1.93


In [20]:
# Ensure datetime dtype
no_cancel_df['test_ordered_dt'] = pd.to_datetime(no_cancel_df['test_ordered_dt'], errors='coerce')

# Weekday = 0–4, Weekend = 5–6
no_cancel_df['day_of_week'] = no_cancel_df['test_ordered_dt'].dt.weekday  # Monday=0
no_cancel_df['day_type'] = no_cancel_df['day_of_week'].apply(lambda x: 'weekday' if x < 5 else 'weekend')

In [21]:
# num unique test codes in no_cancel_df
unique_test_codes_no_cancel = no_cancel_df['test_code'].nunique()
# print(f"Number of unique test codes in no_cancel_df: {unique_test_codes_no_cancel}")

# get unique test codes in no_cancel_df and how many times they appear (e.g. how many unique test codes and their counts)
test_code_counts = no_cancel_df['test_code'].value_counts().to_dict()

test_code_counts_one_or_more = {test_code: count for test_code, count in test_code_counts.items() if count > 1}
# test_code_counts_one_or_more

Calculate relative time for each specimen/test code journey (e.g. ordered = 0, min_result = 0.14 [happens 0.14 hours after being ordered], etc.)

Order of timeline events:
test_ordered_dt: The date/time the order was placed by the ordering physician.
test_collected_dt: The date/time the specimen was collected from the patient.
test_receipt_dt: The date/time the specimen is logged as received in the lab for testing.
test_min/max_result_dt: The date/time the test was noted as resulted (analysis done).
test_mi/max_verified_dt: The date/time the test results were given to the care team for interpretation and action for the patient's care (process complete).
cancellation: can happen at any time after test_ordered_dt, and is reflected in cancellation_dt.


In [22]:
# what is the relative time of each event compared to the first event (test_ordered_dt) for each test_id?
relative_cols = [
    'test_ordered_dt', 'test_collected_dt', 'test_receipt_dt',
    'test_min_resulted_dt', 'test_max_resulted_dt',
    'test_min_verified_dt', 'test_max_verified_dt'
]

# restore test_ordered_dt from timeline in case it was overwritten in a prior run
no_cancel_df['test_ordered_dt'] = no_cancel_df['test_id'].map(
    timeline.set_index('test_id')['test_ordered_dt']
)

# ensure datetime dtype before subtraction
for col in relative_cols:
    no_cancel_df[col] = pd.to_datetime(no_cancel_df[col], utc=True, errors='coerce')

# create new relative columns (do not overwrite original datetime columns)
for col in relative_cols:
    no_cancel_df[col + '_relative'] = (
        no_cancel_df[col] - no_cancel_df['test_ordered_dt']
    ).dt.total_seconds() / 3600  # hours since test_ordered_dt

    # round relative times to 2 decimal places
    no_cancel_df[col + '_relative'] = no_cancel_df[col + '_relative'].round(2)

In [23]:
# show the new relative columns
relative_cols = [col + '_relative' for col in relative_cols]
# no_cancel_df[relative_cols].head()

In [24]:
timeline.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249466 entries, 0 to 249465
Data columns (total 67 columns):
 #   Column                         Non-Null Count   Dtype              
---  ------                         --------------   -----              
 0   test_id                        249466 non-null  object             
 1   AdultER                        997 non-null     datetime64[ns, UTC]
 2   BCSC                           28 non-null      datetime64[ns, UTC]
 3   BCSC -POCT                     3851 non-null    datetime64[ns, UTC]
 4   BCSC-Heme                      38 non-null      datetime64[ns, UTC]
 5   Blood Bank                     1301 non-null    datetime64[ns, UTC]
 6   C1                             55 non-null      datetime64[ns, UTC]
 7   CANHC                          3 non-null       datetime64[ns, UTC]
 8   CENTRIFUGE                     1124 non-null    datetime64[ns, UTC]
 9   CWS                            17848 non-null   datetime64[ns, UTC]
 10  Cancer C

In [25]:
# want to look at day/night shift (like weekend/weekday) - we can use the day_type column we created above
# we can also create a new column for shift based on the hour of test_ordered_dt
no_cancel_df['hour_of_day'] = no_cancel_df['test_ordered_dt'].dt.hour

# if hour is between 7 and 19, it's day shift, otherwise it's night shift (7am-7pm is day shift, 7pm-7am is night shift, assuming 24-hour time)
no_cancel_df['shift'] = no_cancel_df['hour_of_day'].apply(lambda x: 'day' if 7 <= x < 19 else 'night')


Interpreting the relative columns:
- a value of 0 means the event happened at the same time as test_ordered_dt
- apositive value means the event happened after test_ordered_dt
- a negative value would mean the event happened before test_ordered_dt, which shouldn't happen for these events, but could indicate data quality issues if it does

Intepreting a row:
- test_id: 12345_CBC
- test_ordered_dt_relative: 0 (this is the reference point)
- test_collected_dt_relative: 0.083333 (specimen was collected 0.083333 hours after ordering)
- test_receipt_dt_relative: 0.583333 (specimen was received in lab 0.583333 hours after ordering)
- test_min_resulted_dt_relative: 1.283333 (test was resulted 1.283333 hours after ordering)
- test_max_resulted_dt_relative: 1.283333 (same as min resulted in this case)
- test_min_verified_dt_relative: 1.266667 (results were verified 1.266667 hours after ordering) 
- test_max_verified_dt_relative: 1.266667 (same as min_resulted in this case)

All possible locations the test tube could have stopped at (a timestamp in the DF to show a record of the specimen being there)

locations = [
    'AdultER', 'BCSC', 'BCSC -POCT', 'BCSC-Heme', 'Blood Bank',
       'C1', 'CANHC', 'CENTRIFUGE', 'CWS', 'Cancer Cntr', 'CardioV Ctr',
       'Cent-Reflab', 'Chemistry', 'DFLA Remote', 'EAAHC', 'Flow',
       'Hem/Coa/Urn', 'INFTY Rcv', 'IOM_C', 'IOM_H', 'Livonia', 'NCRC HLA',
       'NCRC Immuno', 'NCRC Micro', 'NCRC SpChem', 'NCRC SpQFTB', 'NLNC SP',
       'NOT SENDOUT', 'NoRcv Setup', 'Northv Rmt', 'Northv-poc', 'Northville',
       'P512', 'P612', 'PedsER', 'RIM', 'SENDOUTS', 'SP CoreAuto',
       'Spec. Proc.', 'Taubman1', 'Taubman3', 'UCW', 'UHS', 'West POC',
       'West Recv', 'West Rmote', 'YMBD Remote'
]

In [26]:
# from the data, for each location, find which test codes appear at each location 

locations = [
    'AdultER', 'BCSC', 'BCSC -POCT', 'BCSC-Heme', 'Blood Bank',
       'C1', 'CANHC', 'CENTRIFUGE', 'CWS', 'Cancer Cntr', 'CardioV Ctr',
       'Cent-Reflab', 'Chemistry', 'DFLA Remote', 'EAAHC', 'Flow',
       'Hem/Coa/Urn', 'INFTY Rcv', 'IOM_C', 'IOM_H', 'Livonia', 'NCRC HLA',
       'NCRC Immuno', 'NCRC Micro', 'NCRC SpChem', 'NCRC SpQFTB', 'NLNC SP',
       'NOT SENDOUT', 'NoRcv Setup', 'Northv Rmt', 'Northv-poc', 'Northville',
       'P512', 'P612', 'PedsER', 'RIM', 'SENDOUTS', 'SP CoreAuto',
       'Spec. Proc.', 'Taubman1', 'Taubman3', 'UCW', 'UHS', 'West POC',
       'West Recv', 'West Rmote', 'YMBD Remote'
]

test_codes_by_location = {}

# loop through each location and find the unique test codes that have a timestamp (not null) in that location column (indicating the test went through that location)
for location in locations:

    # if there's a timestamp for that location (i.e. if the test went through that location), then include the test code in the list for that location
    test_codes_by_location[location] = no_cancel_df[no_cancel_df[location].notnull()]['test_code'].unique().tolist()


# get a count of which test codes appear at each location
test_code_counts_by_location = {loc: len(codes) for loc, codes in test_codes_by_location.items()}

In [27]:
# test_code_counts_by_location
# test_codes_by_location

# Visualization Work

### Visulization 1

Visualize (visual, inspectable, graphic timelines) the “average” time series of events for laboratory orders, given a set of dimensional filters (such as “all CBCs ordered in ward CW1”). Method is open to interpretation, and whichever means the students find most readily available.


The dimensional filters we are using are test code, day_type (weekday vs weekend), and shift (day vs night).

We are using these because:
1. test code - filtering by test code allows us to analyze the average time series of events for specific types of tests, which may have different processing workflows and timelines. This can help identify patterns or bottlenecks that are specific to certain tests.
2. day_type - filtering by day_type allows us to compare the average time series of events for orders placed on weekdays versus weekends. This can help identify any differences in processing times or workflows that may be associated with the day of the week, such as staffing levels or resource availability.
3. shift - filtering by shift allows us to compare the average time series of events for orders placed during the day versus night. This can help identify any differences in processing times or workflows that may be associated with the time of day, such as staffing levels, resource availability, or differences in the types of tests ordered during different shifts.

### Visualization 2

An A/B version of the above visualization to compare two mutually exclusive datasets (such as “All CBCs ordered in ward CW1, tests ordered on weekdays vs tests ordered on weekends”).

### Visualization 3

Some way to visualize the likelihood of a named time series event, such as “specimen cancelled due to hemolysis”, in that A/B comparison. --> likelihood of cancellation per specimen (depending on it being either weekday/weekend or day/night shift?)